# 🏥 KUH Ameliyat Vaka Hacmi Tahmini — v5-revised
### LSTM · Facebook Prophet · LightGBM · Weighted Ensemble
**Kaynak:** Bui et al. (2023) — *IISE Transactions on Healthcare Systems Engineering*

---

## 📊 Veri Seti Profili (Gerçek)
| Özellik | Değer |
|---|---|
| **Ameliyat Verisi** | KUH_2020_2026_Ameliyat.xlsx — **68.473 kayıt**, 429 doktor |
| **Tarih Aralığı** | 02 Oca 2020 – 05 Mar 2026 |
| **Doluluk Verisi** | KUH_Hastane_Doluluk_Orani.xlsx — **1.071 gün** (01 Ara 2022 – 07 Kas 2025) |
| **Günlük Vaka** | min=1 · maks=90 · ortalama=33 |
| **Eğitim** | Oca 2020 – Ara 2024 |
| **Validasyon** | Oca 2025 – Ara 2025 |
| **Tahmin + Gerçek** | Oca 2026 – 05 Mar 2026 (gerçek veri mevcut!) |

## Değişiklikler
| # | Alan | Değişiklik |
|---|---|---|
| 1 | **Veri Yükleme** | google.colab.files kaldirildi, dogrudan dosya yolu kullanildi |
| 2 | **Sütun Erisimi** | iloc[:,2] yerine 'Hizmet Kayit Tarihi' ile dogrudan erisim |
| 3 | **Doluluk Bitis** | DOLULUK_END='2025-11-07' (gercek bitis tarihi) |
| 4 | **2026 Gercek Veri** | Oca-05 Mar 2026 gercek ameliyat verisi mevcuttur |
| 5 | **Açiklamalar** | Her hücrede Türkçe açiklama + parametre tavsiyeleri |


---
## 🔧 HÜCRE 1 — Bağimliliği Kur

**Ne yapar?** Google Colab ortaminda varsayilan olarak yüklü olmayan üç kütüphaneyi kurar:
- **`prophet`**: Trend + mevsimsellik + tatil etkileri ile zaman serisi tahmin kütüphanesi.
- **`openpyxl`**: .xlsx dosyalarini okumak için gerekli motor.
- **`lightgbm`**: Histogram tabanli gradient boosting — sklearn GBM'den ~10x hizli.

**Parametre Tavsiyesi:** Bu hücrede değiştirilen parametre yoktur.


In [ ]:
# CELL 1 — Bağimliliklari kur (Colab ortaminda çalistirin)
!pip install prophet openpyxl lightgbm -q
print('Kurulum tamamlandi: prophet, openpyxl, lightgbm')


---
## ⚙️ HÜCRE 2 — Import & Merkezi Tuning Paneli

**Ne yapar?** Tüm kütüphaneleri yukler ve modelin her yerinde kullanilan tek kaynak (single source of truth) parametre panelini tanimlar. Parametreler 5 gruba ayrilmistir:

1. **LSTM**: Pencere boyu, katman genisligi, dropout, öğrenme hizi vs.
2. **Özellik Mühendisligi**: Pandemi siniri, tatil penceresi, EWMA span.
3. **LightGBM**: Agaç sayisi, derinlik, öğrenme hizi, örnekleme oranlari.
4. **Prophet**: Changepoint hassasiyeti, mevsimsellik modu, Fourier bileseni sayisi.
5. **Dönem Sinirlari**: Egitim/validasyon/tahmin tarihleri.

**✅ Önerilen Parametre Değisiklikleri:**

| Parametre | Mevcut | Öneri | Gerekçe |
|---|---|---|---|
| LOOKBACK | 28 | 21 veya 42 | Haftalik döngü için 21, 6 haftalik baglam için 42 |
| LSTM_UNITS | 64 | 96 | 68K kayit daha derin aga izin veriyor |
| BATCH_SIZE | 16 | 32 | GPU ile 32 hem hiz hem genelleme açisindan daha iyi |
| LGB_N_ESTIMATORS | 300 | 500 | Daha fazla agaç + erken durdurma ile optimize |
| PROPHET_CP_SCALE | 0.15 | 0.3 | Pandemi kirilmasi için daha esnek trend |

**🔬 Test Edilmesi Gereken (optimum bilinmiyor):**
- `RECURRENT_DROPOUT`: {0.0, 0.1, 0.2} — RNN içi dropout etkisi veriye göre değisir
- `HUBER_DELTA`: {0.5, 1.0, 2.0} — max vaka=90 ile optimum delta'yi test edin
- `COVID_END`: {'2022-03-01', '2022-06-01', '2022-09-01'} — KUH pandemi son tarihini bilin
- `DOL_EWMA_SPAN`: {3, 7, 14} — doluluk ile ameliyat arasindaki gecikme belirsiz
- `PROPHET_YEARLY_N`: {10, 15, 20} — daha fazla Fourier bileşeni overfitting yaratabilir


In [ ]:
# CELL 2 — Import & Konfigürasyon
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, RepeatVector,
    TimeDistributed, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from prophet import Prophet
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb

# ─── DOSYA YOLLARI (Colab yuklemesi kaldiridi) ───────────────────────────────
DATA_PATH    = 'KUH_2020_2026_Ameliyat.xlsx'      # Gerekirse tam yolu girin
DOLULUK_PATH = 'KUH_Hastane_Doluluk_Orani.xlsx'   # Gerekirse tam yolu girin

# ─── LSTM ────────────────────────────────────────────────────────────────────
SEED              = 42
LOOKBACK          = 28      # TEST: 14, 21, 28, 42, 56
HORIZON           = 7
LSTM_UNITS        = 64      # ONERI: 96
DROPOUT           = 0.3     # ONERI: 0.2 (overfitting yoksa)
RECURRENT_DROPOUT = 0.1     # TEST: 0.0, 0.1, 0.2
HUBER_DELTA       = 1.0     # TEST: 0.5, 1.0, 2.0
EPOCHS            = 200
BATCH_SIZE        = 16      # ONERI: GPU'da 32
LEARNING_RATE     = 3e-4
CLIP_NORM         = 5.0
EARLY_PATIENCE    = 30      # ONERI: 25-40 arasi
REDUCE_FACTOR     = 0.5
REDUCE_PATIENCE   = 15
VAL_SPLIT         = 0.15    # ONERI: 0.10 (veri az ise)

# ─── Özellik Mühendisligi ────────────────────────────────────────────────────
COVID_END         = '2022-06-01'   # TEST: '2022-03-01', '2022-06-01', '2022-09-01'
HOL_WINDOW        = 3
DOL_EWMA_SPAN     = 7             # TEST: 3, 7, 14, 30

# ─── LightGBM ────────────────────────────────────────────────────────────────
LGB_N_ESTIMATORS  = 300    # ONERI: 500
LGB_MAX_DEPTH     = 6      # TEST: 4, 6, 8
LGB_LR            = 0.05   # ONERI: 0.03 (500 agaç ile)
LGB_SUBSAMPLE     = 0.8
LGB_MIN_LEAF      = 5      # TEST: 3, 5, 10
LGB_COLSAMPLE     = 0.8    # TEST: 0.6, 0.8, 1.0

# ─── Prophet ─────────────────────────────────────────────────────────────────
PROPHET_CP_SCALE  = 0.15   # ONERI: 0.3 (pandemi kirilmasi)
PROPHET_SEAS_MODE = 'multiplicative'
PROPHET_SEAS_PRIOR= 10.0
PROPHET_HOL_PRIOR = 15.0
PROPHET_YEARLY_N  = 15     # TEST: 10, 15, 20

# ─── Dönem Sinirlari (veriye göre güncellendi) ───────────────────────────────
TRAIN_END      = '2025-01-01'
VAL_END        = '2026-01-01'
FORECAST_START = '2026-01-01'
FORECAST_END   = '2026-03-31'
DOLULUK_END    = '2025-11-07'  # Gercek doluluk verisinin son günü

np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}')
print(f'GPU: {gpus[0].name if gpus else "Bulunamadi - CPU modu"}')
print(f'LightGBM: {lgb.__version__}')
print()
print('Aktif parametreler:')
print(f'  LSTM    : LOOKBACK={LOOKBACK}  UNITS={LSTM_UNITS}  DROP={DROPOUT}  REC_DROP={RECURRENT_DROPOUT}')
print(f'  Opt     : LR={LEARNING_RATE}  CLIP={CLIP_NORM}  HUBER={HUBER_DELTA}')
print(f'  Ozellik : COVID_END={COVID_END}  HOL_WIN={HOL_WINDOW}  EWMA={DOL_EWMA_SPAN}')
print(f'  LGB     : trees={LGB_N_ESTIMATORS}  depth={LGB_MAX_DEPTH}  lr={LGB_LR}')
print(f'  Prophet : cp={PROPHET_CP_SCALE}  mode={PROPHET_SEAS_MODE}  yearly={PROPHET_YEARLY_N}')


---
## 📂 HÜCRE 2b — Colab Dosya Yükleme

**Ne yapar?** Çalışma ortamını algılar:
- **Google Colab** → `files.upload()` widget'ı açar; yüklenen dosyalar otomatik tanınır.
- **Jupyter / Yerel** → Hücre 2'de tanımlı `DATA_PATH` ve `DOLULUK_PATH` değişkenlerini kullanır, bu hücre sessizce atlanır.

**Parametre notu:** Yükleme sırası önemli değil — dosya adına göre otomatik eşleştirilir.

> ⚠️ Bu hücreyi yalnızca **bir kez** çalıştırın. Tekrar çalıştırırsanız Colab'da yeniden yükleme ekranı açılır.

In [ ]:
# CELL 2b — Colab Dosya Yukle
import sys, io, os

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print('Lutfen iki Excel dosyasini yukleyin:')
    print('  1) KUH_2020_2026_Ameliyat.xlsx')
    print('  2) KUH_Hastane_Doluluk_Orani.xlsx')
    uploaded = files.upload()  # Dosya secici acar

    # Yuklenen dosyalari dogru degiskenlere eslesтир
    for fname in uploaded:
        if 'Ameliyat' in fname or 'ameliyat' in fname:
            DATA_PATH = fname
            print(f'Ameliyat verisi bulundu: {fname}')
        elif 'Doluluk' in fname or 'doluluk' in fname or 'Doluuk' in fname:
            DOLULUK_PATH = fname
            print(f'Doluluk verisi bulundu: {fname}')

    # Kontrol
    missing = []
    if 'DATA_PATH' not in dir() or not os.path.exists(DATA_PATH):
        missing.append('KUH_2020_2026_Ameliyat.xlsx')
    if 'DOLULUK_PATH' not in dir() or not os.path.exists(DOLULUK_PATH):
        missing.append('KUH_Hastane_Doluluk_Orani.xlsx')
    if missing:
        raise FileNotFoundError(f'Eksik dosya(lar): {missing}')
    print('\nTum dosyalar yuklendi. Devam edebilirsiniz.')
else:
    # Yerel / Jupyter ortami
    print('Colab ortami degil — dosya yollari Hucre 2deki degiskenlerden alinacak:')
    print(f'  DATA_PATH    = {DATA_PATH}')
    print(f'  DOLULUK_PATH = {DOLULUK_PATH}')
    for p, label in [(DATA_PATH, 'Ameliyat'), (DOLULUK_PATH, 'Doluluk')]:
        if os.path.exists(p):
            print(f'  ✓ {label} dosyasi bulundu: {p}')
        else:
            print(f'  ✗ {label} dosyasi BULUNAMADI: {p}  ← Hucre 2deki yolu duzeltín')


---
## 📂 HÜCRE 3 — Veri Yükleme

**Ne yapar?** İki Excel dosyasini diskten okur.

**KUH_2020_2026_Ameliyat.xlsx** (68.473 kayit, 9 sütun):
- `Hizmet Kayit Tarihi`: ameliyat tarihi ve saati (sütun ismiyle dogrudan erisim)
- `Primer Doktor Kodu`: ameliyati yapan doktor (429 benzersiz doktor)
- Her satir bir ameliyat — günlük hacime çevrilir.
- Aylik aktif doktor sayisi (num_prov) kapasite proxy'si olarak hesaplanir.

**KUH_Hastane_Doluluk_Orani.xlsx** (1.071 gün):
- Sütunlar: Tarih, Doluluk (0.30 – 0.83 arasi, ort=0.578)
- 01 Ara 2022 – 07 Kas 2025 arasini kapsar

**NOT:** 2026 Oca–05 Mar gercek ameliyat verisi de mevcuttur. Cell 12'de otomatik karsilastirilacaktir.

**Parametre Tavsiyesi:** Veri guncellendiginde DATA_PATH ve DOLULUK_PATH'i Cell 2'de degistirin.


In [ ]:
# CELL 3 — Veri yukle
print('Ameliyat verisi yukleniyor...')
df_raw = pd.read_excel(DATA_PATH)

# Sutun adiyla dogrudan erisim (eski versiyonda iloc[:,2] kullaniliyordu)
df_raw['date'] = pd.to_datetime(df_raw['Hizmet Kayit Tarihi'], errors='coerce').dt.normalize()
df_raw = df_raw.dropna(subset=['date'])

daily_raw  = df_raw.groupby('date').size().reset_index(name='volume')
full_range = pd.date_range(daily_raw['date'].min(), daily_raw['date'].max(), freq='D')
daily = (daily_raw.set_index('date')
                  .reindex(full_range, fill_value=0)
                  .reset_index()
                  .rename(columns={'index': 'date'}))

# Aylik aktif doktor sayisi (kapasite proxy'si)
df_raw['month_key'] = df_raw['date'].dt.to_period('M')
prov_monthly = (df_raw.groupby('month_key')['Primer Doktor Kodu']
                      .nunique().reset_index(name='num_prov'))

print(f'Ameliyat: {len(df_raw):,} kayit yuklendi')
print(f'  Tarih: {daily["date"].min().date()} - {daily["date"].max().date()}')
print(f'  Toplam gun: {len(daily)}  (veri olan: {(daily.volume > 0).sum()})')
print(f'  Gunluk: min={daily.volume.min()}  maks={daily.volume.max()}  ort={daily.volume.mean():.1f}')
print(f'  Benzersiz doktor: {df_raw["Primer Doktor Kodu"].nunique()}')

print()
print('Doluluk verisi yukleniyor...')
df_dol = pd.read_excel(DOLULUK_PATH)
df_dol.columns = ['date', 'doluluk']
df_dol['date']    = pd.to_datetime(df_dol['date']).dt.normalize()
df_dol['doluluk'] = pd.to_numeric(df_dol['doluluk'], errors='coerce')
df_dol = df_dol.dropna().drop_duplicates('date').set_index('date').sort_index()

print(f'Doluluk: {len(df_dol):,} gun yuklendi')
print(f'  Tarih: {df_dol.index.min().date()} - {df_dol.index.max().date()}')
print(f'  Aralik: {df_dol.doluluk.min():.2f} - {df_dol.doluluk.max():.2f}  (ort={df_dol.doluluk.mean():.3f})')


---
## 📊 HÜCRE 4 — Mevsimsel Impütasyon & EDA

**Ne yapar?** Doluluk verisinin eksik oldugu dönemleri doldurur ve 4 grafik üretir.

**Impütasyon Mantigi:**
1. Gercek doluluk verisi yalnizca 01 Ara 2022 – 07 Kas 2025 arasinda mevcut.
2. Eksik günler için **aylik ortalama** kullanilir (Ocak icin Ocak aylarinin ortalamasi).
3. Türetilmis ozellikler: dol_roll7 (7 günlük kayan ort.), dol_delta (günlük degisim).

**EDA Grafikleri:**
- G1: Günlük vaka hacmi (pandemi + validasyon + tahmin dönemleri isaretle)
- G2: Doluluk orani (gercek + impüte)
- G3: Yillik ortalama vaka hacmi
- G4: Doluluk x Vaka korelasyonu (Pearson r)

**TEST:** Pearson r > 0.3 ise doluluk modele anlamli katki sagliyordu demektir. r < 0.1 ise doluluk özelliklerini FEATURE_COLS'den cikarmayi deneyin.

**TEST:** Impütasyon icin monthly_avg.mean() yerine monthly_avg.median() deneyin — medyan aykiri degerlere karsi daha direnclidir.


In [ ]:
# CELL 4 — Mevsimsel imputasyon & EDA

# Aylik ortalama hesapla
dol_monthly_avg = df_dol.copy()
dol_monthly_avg['month'] = dol_monthly_avg.index.month
monthly_avg = dol_monthly_avg.groupby('month')['doluluk'].mean()

# Tüm gün araligi
all_dates = pd.date_range(daily['date'].min(), '2026-03-31', freq='D')
dol_full  = pd.Series(index=all_dates, dtype=float, name='doluluk')
dol_full.update(df_dol['doluluk'])

# Eksik gunleri aylik ortalama ile doldur
for dt in dol_full[dol_full.isna()].index:
    dol_full[dt] = monthly_avg.get(dt.month, monthly_avg.mean())

dol_roll7 = dol_full.rolling(7, min_periods=1, center=True).mean()
dol_delta = dol_full.diff().fillna(0)

daily['doluluk']   = daily['date'].map(dol_full).values
daily['dol_roll7'] = daily['date'].map(dol_roll7).values
daily['dol_delta'] = daily['date'].map(dol_delta).values

n_before = (daily['date'] < '2022-12-01').sum()
n_after  = (daily['date'] > pd.Timestamp(DOLULUK_END)).sum()
print(f'Doluluk imputasyonu:')
print(f'  Gercek veri: 01 Ara 2022 - {DOLULUK_END}')
print(f'  Impute oncesi: {n_before} gun')
print(f'  Impute sonrasi: {n_after} gun')

# EDA Grafikleri
fig, axes = plt.subplots(4, 1, figsize=(16, 14))

axes[0].plot(daily['date'], daily['volume'], color='#3B82F6', lw=0.8, alpha=0.8)
axes[0].axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp(COVID_END),
                alpha=0.08, color='red', label=f'Pandemi (<{COVID_END})')
axes[0].axvspan(pd.Timestamp(TRAIN_END), pd.Timestamp(VAL_END),
                alpha=0.08, color='orange', label='Validasyon 2025')
axes[0].axvspan(pd.Timestamp(FORECAST_START), daily['date'].max(),
                alpha=0.08, color='green', label='Tahmin/Gercek 2026')
axes[0].set_title('Gunluk Ameliyat Vaka Hacmi (2020-2026)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Vaka'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

dol_vis = dol_full[(dol_full.index >= daily['date'].min()) & (dol_full.index <= daily['date'].max())]
axes[1].fill_between(dol_vis.index, dol_vis.values, alpha=0.2, color='#8B5CF6')
axes[1].plot(dol_vis.index, dol_vis.values, color='#8B5CF6', lw=0.8, alpha=0.9, label='Doluluk (tum)')
axes[1].plot(df_dol['doluluk'].index, df_dol['doluluk'].values, color='#4C1D95', lw=1.5, label='Doluluk (gercek)')
axes[1].axvspan(daily['date'].min(), pd.Timestamp('2022-11-30'), alpha=0.08, color='gray', label='Imputasyon')
axes[1].axvspan(pd.Timestamp(DOLULUK_END), daily['date'].max(), alpha=0.08, color='gray')
axes[1].set_title('Hastane Doluluk Orani', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Doluluk'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

daily['year'] = daily['date'].dt.year
yr_avg = daily[daily['volume']>0].groupby('year')['volume'].mean()
bars = axes[2].bar(yr_avg.index, yr_avg.values,
                   color=['#F87171' if y < 2022 else '#3B82F6' for y in yr_avg.index], alpha=0.85)
for bar, val in zip(bars, yr_avg.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Yillik Ortalama Vaka (kirmizi=pandemi)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Ort. Vaka/Gun'); axes[2].grid(axis='y', alpha=0.3)
daily.drop(columns=['year'], inplace=True, errors='ignore')

overlap = daily[(daily['date'] >= '2022-12-01') & (daily['volume'] > 0)]
sc = axes[3].scatter(overlap['doluluk'], overlap['volume'],
                     c=overlap['date'].map(pd.Timestamp.toordinal), cmap='viridis', alpha=0.4, s=20)
if len(overlap) > 1:
    m_, b_ = np.polyfit(overlap['doluluk'], overlap['volume'], 1)
    x_ = np.linspace(overlap['doluluk'].min(), overlap['doluluk'].max(), 100)
    axes[3].plot(x_, m_*x_+b_, color='#EF4444', lw=2, label=f'Trend: y={m_:.1f}x{b_:+.1f}')
corr = overlap['doluluk'].corr(overlap['volume'])
axes[3].set_title(f'Doluluk x Vaka Korelasyonu  (Pearson r = {corr:.3f})', fontsize=12, fontweight='bold')
axes[3].set_xlabel('Doluluk Orani'); axes[3].set_ylabel('Gunluk Vaka')
axes[3].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.colorbar(sc, ax=axes[3], label='Zaman')
if len(overlap) > 1: axes[3].legend()
axes[3].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Doluluk-Vaka Pearson r = {corr:.3f}')
print('  r > 0.3: doluluk anlamli katki sagliyor')
print('  r < 0.1: doluluk ozelliklerini cikarmayi dusunun')


---
## HÜCRE 5 — Türk Tatil Günleri (2019-2026)

**Ne yapar?** İki set tatil tanimlar:
1. **Sabit milli tatiller**: Her yil ayni günde (1 Ocak, 23 Nisan, 1 Mayis, 19 Mayis, 15 Temmuz, 30 Agustos, 29 Ekim)
2. **Dini tatiller**: Ramazan Bayrami + Kurban Bayrami — her yil için manuel girilmis

**NOT:** 2026 Mart icinde Ramazan Bayrami (20-22 Mar) var — tahmin dönemine denk geliyor.

**TEST:** KUH'da bazi tatillerde ameliyat yapiliyorsa (acil), o günleri listeden cikarmak modeli iyilestirebilir. lower_window/upper_window degerlerini Prophet'te degistirerek tatil öncesi/sonrasi etki uzunlugunu test edin.


In [ ]:
# CELL 5 — Türk tatil günleri (2019-2026)
TR_HOL = set()
for y in range(2019, 2027):
    for m, d in [(1,1),(4,23),(5,1),(5,19),(7,15),(8,30),(10,29)]:
        TR_HOL.add(pd.Timestamp(f'{y}-{m:02d}-{d:02d}'))

RELIGIOUS = {
    # Ramazan Bayrami
    pd.Timestamp('2020-05-24'), pd.Timestamp('2020-05-25'), pd.Timestamp('2020-05-26'),
    pd.Timestamp('2021-05-13'), pd.Timestamp('2021-05-14'), pd.Timestamp('2021-05-15'),
    pd.Timestamp('2022-05-02'), pd.Timestamp('2022-05-03'), pd.Timestamp('2022-05-04'),
    pd.Timestamp('2023-04-21'), pd.Timestamp('2023-04-22'), pd.Timestamp('2023-04-23'),
    pd.Timestamp('2024-04-10'), pd.Timestamp('2024-04-11'), pd.Timestamp('2024-04-12'),
    pd.Timestamp('2025-03-30'), pd.Timestamp('2025-03-31'), pd.Timestamp('2025-04-01'),
    pd.Timestamp('2026-03-20'), pd.Timestamp('2026-03-21'), pd.Timestamp('2026-03-22'),  # Tahmin doneminde!
    # Kurban Bayrami
    pd.Timestamp('2020-07-31'), pd.Timestamp('2020-08-01'), pd.Timestamp('2020-08-02'),
    pd.Timestamp('2021-07-20'), pd.Timestamp('2021-07-21'), pd.Timestamp('2021-07-22'),
    pd.Timestamp('2022-07-09'), pd.Timestamp('2022-07-10'), pd.Timestamp('2022-07-11'),
    pd.Timestamp('2023-06-28'), pd.Timestamp('2023-06-29'), pd.Timestamp('2023-06-30'),
    pd.Timestamp('2024-06-17'), pd.Timestamp('2024-06-18'), pd.Timestamp('2024-06-19'),
    pd.Timestamp('2025-06-06'), pd.Timestamp('2025-06-07'), pd.Timestamp('2025-06-08'),
}
ALL_HOL = TR_HOL | RELIGIOUS
print(f'Toplam tatil gunu: {len(ALL_HOL)} (2019-2026)')
print('Tahmin donemindeki tatiller (Oca-Mar 2026):')
for h in sorted(ALL_HOL):
    if pd.Timestamp(FORECAST_START) <= h <= pd.Timestamp(FORECAST_END):
        print(f'  {h.date()}  ({h.day_name()})')


---
## HÜCRE 6 — Özellik Mühendisligi

**Ne yapar?** Ham tarih ve hacim verisinden modelin öğrenebilecegi ozellikler üretir. Bu adim tahmin kalitesini en cok etkileyen adimdir.

**Özellik Gruplari:**
| Grup | Ozellikler | Aciklama |
|---|---|---|
| Zaman | day_of_year, week_of_year, year | Dogrusal zaman bilgisi |
| Döngüsel | sin/cos_dayofweek, sin/cos_month, sin/cos_quarter | Haftalik/aylik/ceyreklik devre |
| Tatil | is_holiday, is_weekend, near_hol_{-3..+3} | Tatil ve yakin günler |
| Kapasite | num_prov | Aylik aktif doktor sayisi |
| Pandemi | is_covid | COVID dönemi binary bayragi |
| Anomali | is_anomaly | IsolationForest ile tespit edilen aykiri günler |
| Gecikme | lag_3,7,14,28,60,90 | Gectigi günlerdeki vaka sayisi |
| Yillik | same_week_ly | Gecen yil ayni hafta (pandemi duzeltmeli) |
| Kayan ort. | roll_7, roll_14, roll_28 | Yakin gecmis ortalamalari |
| Doluluk | doluluk, dol_roll7, dol_delta, dol_ewma | Ham + türetilmis |

**TEST:** lag_60 ve lag_90'in katkisini ölcmek icin FEATURE_COLS'dan cikarin ve R2 farkina bakin.
**TEST:** contamination=0.05 (aykiri deger esigi) — pandemi döneminde 0.08 deneyin.
**TEST:** is_anomaly özelligini cikarin — R2 degisimi düsükse anlamlilik katkisi yok demektir.


In [ ]:
# CELL 6 — Özellik mühendisligi

def engineer_features(df):
    df = df.copy()
    d  = pd.to_datetime(df['date'])

    # Zaman bilesenleri
    df['day_of_year']  = d.dt.dayofyear
    df['week_of_year'] = d.dt.isocalendar().week.astype(int)
    df['year']         = d.dt.year

    # Döngüsel kodlama: sin/cos ile Ocak-Aralik kopukluğunu yok et
    for feat, period in [('dayofweek', 7), ('month', 12), ('quarter', 4)]:
        v = getattr(d.dt, feat)
        df[f'sin_{feat}'] = np.sin(2 * np.pi * v / period)
        df[f'cos_{feat}'] = np.cos(2 * np.pi * v / period)

    df['is_holiday'] = d.isin(ALL_HOL).astype(int)
    df['is_weekend']  = (d.dt.dayofweek >= 5).astype(int)

    # Tatil yakinlik penceresi
    for off in range(-HOL_WINDOW, HOL_WINDOW + 1):
        if off == 0: continue
        shifted = {h + pd.Timedelta(days=off) for h in ALL_HOL}
        df[f'near_hol_{off:+d}'] = d.isin(shifted).astype(int)

    # Aylik aktif doktor sayisi
    mk = d.dt.to_period('M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    df['num_prov'] = mk.map(pm).ffill().bfill()

    # Pandemi bayragi
    df['is_covid'] = ((d >= '2020-01-01') & (d < COVID_END)).astype(int)

    # Anomali tespiti: contamination=0.05 -> verinin %5'i aykiri
    # TEST: contamination=0.03 veya 0.08
    iso = IsolationForest(contamination=0.05, random_state=SEED)
    df['is_anomaly'] = (iso.fit_predict(df[['volume']]) == -1).astype(int)

    # Gecikme ozellikleri
    for lag in [3, 7, 14, 28, 60, 90]:
        df[f'lag_{lag}'] = df['volume'].shift(lag).fillna(0)

    # Gecen yil ayni hafta (364 gun = 52 tam hafta)
    lag_1yr = df['volume'].shift(364)
    lag_2yr = df['volume'].shift(364 * 2)
    lag_3yr = df['volume'].shift(364 * 3)
    df['same_week_ly'] = lag_1yr.fillna(lag_2yr).fillna(lag_3yr).fillna(0)
    # Pandemi duzeltmesi: gercek pandemi dusuk degerlerini 2 yil oncesiyle yukselt
    covid_mask     = (df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')
    covid_prev_low = df['same_week_ly'] < 20
    df.loc[covid_mask & covid_prev_low, 'same_week_ly'] = lag_2yr[covid_mask & covid_prev_low].fillna(0)

    # Kayan ortalama (shift(1) -> bugunü dahil etme)
    for win in [7, 14, 28]:
        df[f'roll_{win}'] = df['volume'].shift(1).rolling(win, min_periods=1).mean().fillna(0)

    # EWMA doluluk: son gunlere üstel agirlik
    df['dol_ewma'] = df['doluluk'].ewm(span=DOL_EWMA_SPAN, adjust=False).mean()

    return df

daily = engineer_features(daily)

hol_cols = [f'near_hol_{off:+d}' for off in range(-HOL_WINDOW, HOL_WINDOW + 1) if off != 0]

FEATURE_COLS = (
    ['volume',
     'day_of_year', 'week_of_year',
     'sin_dayofweek', 'cos_dayofweek',
     'sin_month',    'cos_month',
     'sin_quarter',  'cos_quarter',
     'is_holiday', 'is_weekend', 'is_anomaly']
    + hol_cols
    + ['num_prov', 'is_covid',
       'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_60', 'lag_90',
       'same_week_ly',
       'roll_7', 'roll_14', 'roll_28',
       'doluluk', 'dol_roll7', 'dol_delta', 'dol_ewma']
)

print(f'Ozellik muhendisligi: {len(FEATURE_COLS)} ozellik')
print(f'  Anomali: {daily["is_anomaly"].sum()} gun ({daily["is_anomaly"].sum()/len(daily)*100:.1f}%)')
print(f'  COVID dönemi: {daily["is_covid"].sum()} gun')
print(f'  Tatil gunu: {daily["is_holiday"].sum()}')
print(f'  Tatil penceresi: +/-{HOL_WINDOW} gun ({len(hol_cols)} sutun)')


---
## HÜCRE 7 — Egitim/Validasyon Bölünmesi & Ölçekleme

**Ne yapar?** Veriyi zamana göre böler ve LSTM icin dizi matrisleri olusturur.

**Neden RobustScaler?**
- MinMaxScaler: [min, max] arasina normalize eder. Pandemi dönemi degerler çok küçük oldugunda diger degerler sikisir.
- RobustScaler: Medyan ve IQR (çeyrekler arasi aralik) kullanir. KUH verisi icin pandemi günleri aykiri deger olusturur — RobustScaler açik ara daha iyi.

**Dizi Yapisi:**
- X: [batch, LOOKBACK=28, n_features] — son 28 günün tüm özellikleri
- y: [batch, HORIZON=7] — sonraki 7 günün vaka sayisi

**TEST:** HORIZON=14 ile 2 haftalik tahmin deneyin — R2 düserse kisa ufuk daha uygun.
**TEST:** StandardScaler ile karsilastirin.


In [ ]:
# CELL 7 — Egitim/Validasyon bolunmesi & olcekleme

train_df = daily[daily['date'] <  TRAIN_END].copy()  # Oca 2020 - Ara 2024
val_df   = daily[(daily['date'] >= TRAIN_END) & (daily['date'] < VAL_END)].copy()  # 2025

# RobustScaler: medyan+IQR bazli, pandemi aykiriliklarina dayanikli
scaler   = RobustScaler()
train_sc = scaler.fit_transform(train_df[FEATURE_COLS].values)
val_sc   = scaler.transform(val_df[FEATURE_COLS].values)
all_sc   = scaler.transform(daily[FEATURE_COLS].values)

def inv_vol(sv):
    """Olcekli hacim degerini orijinal olcege cevir."""
    d = np.zeros((len(sv), len(FEATURE_COLS)))
    d[:, 0] = sv
    return scaler.inverse_transform(d)[:, 0]

def make_sequences(arr, lb=LOOKBACK, hz=HORIZON):
    """[n_days, n_features] dizisini LSTM icin [n_seq, lb, n_features] haline getir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb])
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr, y_tr = make_sequences(train_sc)
X_va, y_va = make_sequences(np.concatenate([train_sc[-LOOKBACK:], val_sc]))

print(f'Egitim : {len(train_df):,} gun  ({train_df["date"].min().date()} - {train_df["date"].max().date()})')
print(f'Val    : {len(val_df):,} gun   ({val_df["date"].min().date()} - {val_df["date"].max().date()})')
print(f'LSTM dizi - Egitim: {len(X_tr):,}  Val: {len(X_va):,}')
print(f'Sekil: X={X_tr.shape}  y={y_tr.shape}')
print(f'Scaler: merkez={scaler.center_[0]:.2f}  olcek={scaler.scale_[0]:.2f}')


---
## HÜCRE 8 — MODEL 1: Encoder-Decoder Seq2Seq LSTM

**Ne yapar?** Zaman serisi tahmini icin BiLSTM tabanli seq2seq mimari kurar ve egitir.

**Mimari Özeti:**
```
Encoder Giris: [batch, 28, ~43 ozellik]
  BiLSTM (64x2) -> cift yonlu oruntu ogrenme
  LSTM (64) -> gizli durum [h, c]
  RepeatVector(7) -> h'yi 7 kez tekrarla
  Decoder LSTM-1 (64) -> [h,c] ile baslat
  Decoder LSTM-2 (32) -> daha derin kod cozme
  TimeDistributed Dense(1) -> her adim icin tahmin
Cikis: [batch, 7]
```

**Loss: Huber(delta=1.0)** — Büyük hatalara daha az ceza. Max vaka=90 ile pandemi gunleri icin dengeli.

**✅ Öneriler:**
- LSTM_UNITS=64 -> 96: 68K kayit daha derin aga izin veriyor
- BATCH_SIZE=16 -> 32: GPU ile hem hiz hem genelleme icin
- EARLY_PATIENCE=30 -> 25: Egitim cok uzun suruyorsa

**🔬 Test Edilmesi Gerekenler:**
- RECURRENT_DROPOUT: {0.0, 0.1, 0.2} — RNN icindeki dropout etkisi veriye özel
- HUBER_DELTA: {0.5, 1.0, 2.0} — aykiri degerlere tolerans seviyesi
- Decoder LSTM-2'yi kaldirarak basit modeli test edin


In [ ]:
# CELL 8 — MODEL 1: Encoder-Decoder Seq2Seq LSTM

def build_lstm(n_features):
    enc_in = Input(shape=(LOOKBACK, n_features), name='encoder_input')

    # BiLSTM: ileri+geri yönde oruntu ogrenme
    x = Bidirectional(
            LSTM(LSTM_UNITS, return_sequences=True,
                 dropout=DROPOUT,
                 recurrent_dropout=RECURRENT_DROPOUT),
            name='bi_lstm'
        )(enc_in)

    # Encoder LSTM: son gizli durum [h, c] encoder bilgisini sikistirir
    _, h, c = LSTM(LSTM_UNITS, return_state=True,
                   dropout=DROPOUT,
                   recurrent_dropout=RECURRENT_DROPOUT,
                   name='encoder_lstm')(x)

    # Decoder: encoder gizli durumundan HORIZON adim cözümle
    dec  = RepeatVector(HORIZON, name='repeat')(h)
    dec  = LSTM(LSTM_UNITS, return_sequences=True,
                dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT,
                name='decoder_lstm_1')(dec, initial_state=[h, c])
    # 2. Decoder katmani (TEST: bu satiri kaldirarak basit modeli deneyin)
    dec  = LSTM(LSTM_UNITS // 2, return_sequences=True,
                dropout=DROPOUT, name='decoder_lstm_2')(dec)
    out  = TimeDistributed(Dense(1), name='output')(dec)
    out  = tf.keras.layers.Reshape((HORIZON,))(out)

    model = Model(enc_in, out, name='seq2seq_bilstm_v5')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=CLIP_NORM
        ),
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA)
    )
    return model

lstm_model = build_lstm(len(FEATURE_COLS))
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_PATIENCE, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR,
                      patience=REDUCE_PATIENCE, min_lr=1e-7, verbose=1),
]

print(f'\nLSTM egitiliyor... (GPU ~5-12 dk, CPU ~40-60 dk)')
history = lstm_model.fit(
    X_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=callbacks,
    verbose=1
)
print(f'LSTM tamamlandi ({len(history.history["loss"])} epoch)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#3B82F6', lw=1.5, label='Egitim')
ax.plot(history.history['val_loss'], color='#F59E0B', lw=1.5, ls='--', label='Val')
ax.set_title(f'LSTM Kayip Egrisi (Huber delta={HUBER_DELTA})', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Kayip')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Overfitting kontrolü
ratio = history.history['val_loss'][-1] / max(history.history['loss'][-1], 1e-9)
if ratio > 2.0:
    print('UYARI: Overfitting var -> DROPOUT artirin veya LSTM_UNITS azaltin')
elif ratio < 1.1:
    print('Iyi fit — overfitting yok')
else:
    print('Hafif overfitting — kabul edilebilir')


---
## HÜCRE 9 — LSTM Dogrudan Tahmin

**Ne yapar?** Direct forecast stratejisiyle validasyon (2025) ve tahmin (2026) dönemlerini hesaplar.

**Direct Forecast:** Her tahmin adiminda gercek gecmis veriyi kullanir. Modelin kendi hatasi bir sonraki adima girmez — bu kapali döngü (closed-loop) tahminden cok daha yüksek dogruluk saglar.

**2026 Tahmini:** Tüm bilinen gercek veri (2020-2025) baglam olarak kullanilir.

**TEST:** HORIZON'u 14'e cikararak 2 haftalik tahmin deneyin.


In [ ]:
# CELL 9 — LSTM dogrudan tahminleri

def direct_forecast_lstm(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
        n_target     = len(target_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        if mode == 'val':
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            real_past = all_sc
        seed  = real_past[-LOOKBACK:]
        x_in  = seed.reshape(1, LOOKBACK, len(FEATURE_COLS))
        ps    = lstm_model.predict(x_in, verbose=0)[0]
        pv    = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lstm': round(float(p))})

    return pd.DataFrame(results)

fcast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')

print('LSTM validasyon tahmini (2025)...')
lstm_val  = direct_forecast_lstm(mode='val')
print('LSTM 2026 tahmini...')
lstm_test = direct_forecast_lstm(mode='test')
print(f'LSTM: {len(lstm_val)} val, {len(lstm_test)} tahmin gunu')


---
## HÜCRE 10 — MODEL 2: Facebook Prophet

**Ne yapar?** Trend + mevsimsellik + tatil + dis degiskenler ile zaman serisi modeli kurar.

**Multiplicative vs Additive:**
- additive: mevsimsel sapma sabit büyüklükte.
- multiplicative: sapma trende oranli büyür. KUH gibi pandemi sonrasi büyüyen hastanelerde multiplicative daha gercekci.

**Regressor'lar:**
- num_prov: aylik aktif doktor — kapasite artisini bildirir
- is_covid: pandemi dönemini ayrrica ele alir
- doluluk: hastane dolulugu — yüksek doluluk dönemi daha fazla ameliyat

**changepoint_prior_scale:**
- Düsük (0.05): trend yavash degisir, pandemi kirilmasini kacirabilir
- Yüksek (0.5): çok esnek, overfitting riski
- KUH icin öneri: 0.3

**TEST:** PROPHET_YEARLY_N: {10, 15, 20} — daha fazla Fourier bileşeni overfitting yaratabilir.
**TEST:** seasonality_mode='additive' ile karsilastirin.


In [ ]:
# CELL 10 — MODEL 2: Facebook Prophet
print('Prophet egitiliyor...')

hol_df = pd.DataFrame([
    {'holiday': 'tr_tatil', 'ds': h, 'lower_window': -2, 'upper_window': 2}
    for h in ALL_HOL
])

def get_prov(dt):
    mk = pd.Period(pd.Timestamp(dt), 'M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    return float(pm.get(mk, pm.iloc[-1]))

def get_doluluk(dt):
    ts = pd.Timestamp(dt).normalize()
    return float(dol_full.get(ts, dol_full.mean()))

prophet_model = Prophet(
    seasonality_mode=PROPHET_SEAS_MODE,
    weekly_seasonality=True,
    yearly_seasonality=PROPHET_YEARLY_N,
    daily_seasonality=False,
    holidays=hol_df,
    changepoint_prior_scale=PROPHET_CP_SCALE,
    seasonality_prior_scale=PROPHET_SEAS_PRIOR,
    holidays_prior_scale=PROPHET_HOL_PRIOR,
)
prophet_model.add_regressor('num_prov', standardize=True)
prophet_model.add_regressor('is_covid', standardize=False)
prophet_model.add_regressor('doluluk',  standardize=True)

train_prophet = train_df[['date','volume','is_covid']].rename(columns={'date':'ds','volume':'y'}).copy()
train_prophet['num_prov'] = train_prophet['ds'].apply(get_prov)
train_prophet['doluluk']  = train_prophet['ds'].apply(get_doluluk)
prophet_model.fit(train_prophet)

n_future = len(val_df) + len(fcast_dates)
future   = prophet_model.make_future_dataframe(periods=n_future, freq='D')
future['num_prov'] = future['ds'].apply(get_prov)
future['is_covid'] = ((future['ds'] >= '2020-01-01') & (future['ds'] < COVID_END)).astype(int)
future['doluluk']  = future['ds'].apply(get_doluluk)
fc_prophet = prophet_model.predict(future)

prop_val = (fc_prophet[fc_prophet['ds'].isin(val_df['date'])]
            [['ds','yhat']].rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_val['pred_prophet'] = np.clip(prop_val['pred_prophet'], 0, None).round()

prop_test = (fc_prophet[fc_prophet['ds'].isin(fcast_dates)]
             [['ds','yhat']].rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_test['pred_prophet'] = np.clip(prop_test['pred_prophet'], 0, None).round()

print('Prophet tamamlandi')
fig = prophet_model.plot_components(fc_prophet)
fig.suptitle('Prophet Bilesenleri (2020-2026)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## HÜCRE 11 — MODEL 3: LightGBM

**Ne yapar?** Her HORIZON adimi icin ayri bir gradient boosting modeli egitir.

**Düzlestirme (Flatten):**
```
LSTM girisi: [1, 28 gun, 43 ozellik]  (3D tensor)
LGB girisi:  [1, 28x43 = 1204 ozellik] (1D vektor)
```

**LightGBM Avantaji:**
- Histogram tabanli: sklearn GBM'den ~10x hizli
- colsample_bytree: her agac icin rastgele ozellik alt kumesi — overfitting azaltir

**✅ Öneriler:**
- LGB_N_ESTIMATORS=300 -> 500 ve LGB_LR=0.05 -> 0.03: daha fazla agac + kucuk adim
- LGB_MAX_DEPTH=6: iyi deger; 8 de deneyin ama overfitting izleyin

**🔬 Test Edilmesi Gerekenler:**
- LGB_COLSAMPLE=0.8 -> 0.6: daha fazla rastgelelik, overfitting azalabilir
- LGB_MIN_LEAF=5 -> 10: daha büyük yapraklar, daha iyi genelleme
- Ozellik önemi grafigi ekleyin (Cell 11 sonundaki kod ile)


In [ ]:
# CELL 11 — MODEL 3: LightGBM
print('LightGBM egitiliyor...')

def make_seq_flat(arr, lb=LOOKBACK, hz=HORIZON):
    """Dizi matrisini LGB icin 1D vektore duzlestir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb].flatten())
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr_f, y_tr_f = make_seq_flat(train_sc)

lgb_models = []
for step in range(HORIZON):
    m = lgb.LGBMRegressor(
        n_estimators     = LGB_N_ESTIMATORS,
        max_depth        = LGB_MAX_DEPTH,
        learning_rate    = LGB_LR,
        subsample        = LGB_SUBSAMPLE,
        subsample_freq   = 1,
        colsample_bytree = LGB_COLSAMPLE,
        min_child_samples= LGB_MIN_LEAF,
        random_state     = SEED,
        n_jobs           = -1,
        verbose          = -1
    )
    m.fit(X_tr_f, y_tr_f[:, step])
    lgb_models.append(m)

print(f'LightGBM tamamlandi ({HORIZON} model x {LGB_N_ESTIMATORS} agac)')

def direct_forecast_lgb(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = fcast_dates
        n_target     = len(fcast_dates)
    results = []
    dates = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))
    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        real_past = np.vstack([train_sc, val_sc[:i]]) if mode == 'val' else all_sc
        flat = real_past[-LOOKBACK:].flatten().reshape(1, -1)
        ps   = np.array([m.predict(flat)[0] for m in lgb_models])
        pv   = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lgb': round(float(p))})
    return pd.DataFrame(results)

gbm_val  = direct_forecast_lgb(mode='val')
gbm_test = direct_forecast_lgb(mode='test')
gbm_val['pred_gbm']  = gbm_val['pred_lgb']
gbm_test['pred_gbm'] = gbm_test['pred_lgb']
print(f'LightGBM: {len(gbm_val)} val, {len(gbm_test)} tahmin gunu')

# En onemli 10 ozellik (adim 0)
fi = lgb_models[0].feature_importances_
top_idx = np.argsort(fi)[::-1][:10]
print('\nLGB Adim-1 En Onemli 10 Ozellik:')
for rank, idx in enumerate(top_idx):
    day_i  = idx // len(FEATURE_COLS)
    feat_i = idx %  len(FEATURE_COLS)
    print(f'  #{rank+1:2d}: Gun{day_i+1} - {FEATURE_COLS[feat_i]:<25} (onem={fi[idx]:.0f})')


---
## HÜCRE 12 — Metrikler, 2026 Gercek Karsilasma & Agirlikli Ensemble

**Ne yapar?** Üç modeli R2, RMSE, MAE ile degerlendirir ve agirlikli ensemble olusturur.

**Metrikler:**
- R2: 1.0=mukemmel, 0.0=ortalama kadar, <0=ortalamadasn kötü. Hedef: >0.7
- RMSE: vaka cinsinden mutlak hata
- MAE: aykiri degerlere daha direncli hata ölcümü

**Agirlikli Ensemble:**
```
agirlik_i = max(R2_i, 0) / toplam_pozitif_R2
tahmin = sum(agirlik_i * tahmin_i)
```

**2026 Gercek Karsilasma:** Ameliyat verisi 05 Mar 2026'ya kadar mevcut — otomatik karsilastirilir.

**TEST:** Ensemble icin medyan toplamayi deneyin:
pred_ensemble = np.median([pred_lstm, pred_prophet, pred_gbm], axis=0)


In [ ]:
# CELL 12 — Metrikler & Agirlikli Ensemble

def compute_metrics(y_true, y_pred):
    yt = np.array(y_true); yp = np.array(y_pred)
    m  = yt > 0
    if m.sum() == 0: return 0.0, 0.0, 0.0
    return (r2_score(yt[m], yp[m]),
            np.sqrt(mean_squared_error(yt[m], yp[m])),
            mean_absolute_error(yt[m], yp[m]))

def build_comparison(actual_df, *pred_dfs):
    merged = actual_df[['date','volume']].copy()
    for pdf in pred_dfs:
        merged = merged.merge(pdf, on='date', how='left')
    return merged.fillna(0)

val_comp = build_comparison(val_df, lstm_val, prop_val, gbm_val)

# 2026 gercek verisi (versetinde Oca-05 Mar mevcut)
actual_2026 = daily[daily['date'] >= pd.Timestamp(FORECAST_START)][['date','volume']]
test_comp = pd.DataFrame({'date': fcast_dates})
for pdf in [lstm_test, prop_test, gbm_test]:
    test_comp = test_comp.merge(pdf, on='date', how='left')
test_comp = test_comp.merge(actual_2026, on='date', how='left').fillna(0)
test_comp['is_weekend'] = pd.DatetimeIndex(test_comp['date']).dayofweek >= 5
test_comp['is_holiday'] = pd.DatetimeIndex(test_comp['date']).isin(ALL_HOL)

print('='*60)
print('  Validasyon Performansi — 2025')
print(f"  {'Model':<16} {'R2':>6} {'RMSE':>7} {'MAE':>7}  Durum")
print('  ' + '-'*52)
for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    durum = 'Iyi (R2>0.7)' if r2 >= 0.7 else ('Orta' if r2 >= 0.4 else 'Dusuk')
    print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}  {durum}")
print(f"  {'Makale LSTM':<16} {'0.855':>6} {'2.017':>7} {'1.104':>7}  (Bui 2023 ref.)")
print('='*60)

# Agirlikli Ensemble
R2_lstm    = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[0]
R2_prophet = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[0]
R2_gbm     = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[0]
w_raw = np.array([max(R2_lstm, 0), max(R2_prophet, 0), max(R2_gbm, 0)])
if w_raw.sum() == 0: w_raw = np.ones(3)
weights = w_raw / w_raw.sum()
w_lstm, w_prophet, w_gbm = weights
print(f'\nEnsemble agirliklar: LSTM={w_lstm:.3f}  Prophet={w_prophet:.3f}  LGB={w_gbm:.3f}')

val_comp['pred_ensemble'] = (
    w_lstm * val_comp['pred_lstm'] +
    w_prophet * val_comp['pred_prophet'] +
    w_gbm * val_comp['pred_gbm']
).round()
test_comp['pred_ensemble'] = (
    w_lstm * test_comp['pred_lstm'] +
    w_prophet * test_comp['pred_prophet'] +
    w_gbm * test_comp['pred_gbm']
).round()

r2e, rme, mae_e = compute_metrics(val_comp['volume'], val_comp['pred_ensemble'])
print(f'Ensemble Val: R2={r2e:.3f}  RMSE={rme:.1f}  MAE={mae_e:.1f}')

has_actual = test_comp['volume'].sum() > 0
if has_actual:
    n_actual = (test_comp['volume'] > 0).sum()
    print(f'\n2026 Gercek Karsilasma ({n_actual} gun mevcut):')
    for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),
                      ('pred_gbm','LightGBM'),('pred_ensemble','Ensemble')]:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        print(f'  {name:<12}: R2={r2:.3f}  RMSE={rm:.1f}  MAE={ma:.1f}')

# Ocak 2026 detay tablosu
jan = test_comp[pd.DatetimeIndex(test_comp['date']).month == 1].copy()
jan['day'] = pd.DatetimeIndex(jan['date']).day_name()
print('\nOcak 2026 Tahminleri:')
for _, r in jan.iterrows():
    note = 'Tatil' if r['is_holiday'] else ('Hft.sonu' if r['is_weekend'] else '')
    print(f"  {str(r['date'].date())}  {r['day']:<10} LSTM={int(r['pred_lstm']):3d}  "
          f"Prophet={int(r['pred_prophet']):3d}  GBM={int(r['pred_gbm']):3d}  "
          f"Ens={int(r['pred_ensemble']):3d}" +
          (f"  Gercek={int(r['volume']):3d}" if has_actual else '') + f'  {note}')


---
## HÜCRE 13 — Ana Görselleştirme (6 Panel)

**Ne yapar?** Tüm analiz sonuçlarini 6 panel grafik halinde birlestirir.

| Panel | Icerik |
|---|---|
| Satir 0 | 2025 validasyon: tüm modeller vs gercek |
| Satir 1 | 2026 tahmin (Oca-Mar) + gercek bar olarak |
| Satir 2 | Scatter (gercek vs tahmin), 3 model ayri |
| Satir 3 | Hata dagilim histogrami |
| Satir 4 | Model karsilasma (R2, RMSE, MAE bar grafigi) |
| Satir 5 | Doluluk + vaka hacmi + korelasyon scatter |

**TEST:** figsize=(18,34) -> (20,40) buyutun veya dpi=150 -> 200 artirin.


In [ ]:
# CELL 13 — Ana görsellestime
COLS = {'LSTM':'#3B82F6', 'Prophet':'#F59E0B', 'GBM':'#10B981', 'Ensemble':'#8B5CF6', 'Actual':'#1E293B'}

fig = plt.figure(figsize=(18, 34), facecolor='#F8FAFC')
gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.52, wspace=0.32, top=0.95, bottom=0.03, left=0.06, right=0.97)

# Satir 0: Validasyon 2025
ax0 = fig.add_subplot(gs[0, :]); ax0.set_facecolor('white')
ax0.plot(val_df['date'], val_df['volume'], color=COLS['Actual'], lw=2, label='Gercek', zorder=5)
for col, name, ls in [('pred_lstm','LSTM','--'),('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),('pred_ensemble','Ensemble','-')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax0.plot(val_df['date'], val_comp[col], color=COLS.get(name, COLS['GBM']), lw=1.7, ls=ls.strip(),
             label=f'{name:<12} R2={r2:.3f}  RMSE={rm:.1f}')
ax0.set_title('Validasyon 2025: Tum Modeller vs Gercek', fontsize=12, fontweight='bold')
ax0.set_ylabel('Gunluk Vaka'); ax0.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax0.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax0.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax0.legend(fontsize=9, ncol=2, prop={'family':'monospace'}); ax0.grid(alpha=0.2)

# Satir 1: 2026 Tahminleri
ax1 = fig.add_subplot(gs[1, :]); ax1.set_facecolor('white')
for _, r in test_comp[test_comp['is_weekend'] | test_comp['is_holiday']].iterrows():
    ax1.axvspan(r['date'] - pd.Timedelta(hours=12), r['date'] + pd.Timedelta(hours=12), alpha=0.07, color='gray')
if has_actual:
    ax1.bar(test_comp['date'], test_comp['volume'], color='#CBD5E1', width=0.85, label='Gercek', zorder=2)
for col, name, ls in [('pred_lstm','LSTM','--'),('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),('pred_ensemble','Ensemble','-')]:
    lbl = name
    if has_actual:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        lbl = f'{name} R2={r2:.3f}'
    ax1.plot(test_comp['date'], test_comp[col], color=COLS.get(name, COLS['GBM']), lw=2.2, ls=ls.strip(), label=lbl)
ax1.set_title('Tahmin — Oca-Mar 2026', fontsize=12, fontweight='bold'); ax1.set_ylabel('Gunluk Vaka')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax1.legend(fontsize=9, ncol=4, prop={'family':'monospace'}); ax1.grid(alpha=0.2)

# Satir 2: Scatter
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[2, ci]); ax.set_facecolor('white')
    mw = val_comp[val_comp['volume'] > 0]
    ax.scatter(mw['volume'], mw[col], color=COLS[name], alpha=0.55, s=35)
    mn = max(0, min(mw['volume'].min(), mw[col].min()) - 3)
    mx = max(mw['volume'].max(), mw[col].max()) + 3
    ax.plot([mn,mx],[mn,mx], '--', color='#94A3B8', lw=1.2, label='Mukemmel fit')
    if len(mw) > 1:
        m_, b_ = np.polyfit(mw['volume'], mw[col], 1)
        ax.plot([mn,mx],[m_*mn+b_, m_*mx+b_], '-', color='#EF4444', lw=1.5)
    r2, rm, ma = compute_metrics(mw['volume'], mw[col])
    ax.text(0.06, 0.83, f'R2={r2:.3f}\nRMSE={rm:.1f}\nMAE={ma:.1f}',
            transform=ax.transAxes, fontsize=9, family='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#DBEAFE', alpha=0.85))
    ax.set_title(f'{name} — Val 2025', fontsize=10, fontweight='bold'); ax.grid(alpha=0.2)

# Satir 3: Hata Dagilimi
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[3, ci]); ax.set_facecolor('white')
    mw = val_comp[val_comp['volume'] > 0]
    err = mw[col].values - mw['volume'].values
    ax.hist(err, bins=28, color=COLS[name], alpha=0.75, edgecolor='white')
    ax.axvline(0, color='black', lw=1.5, ls='--', label='Sifir hata')
    ax.axvline(err.mean(), color='#EF4444', lw=1.5, label=f'Ort={err.mean():+.1f}')
    ax.set_title(f'{name} — Hata Dagilimi', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

# Satir 4: Performans Bar
ax4 = fig.add_subplot(gs[4, :]); ax4.set_facecolor('white')
mnames = ['LSTM\n(Enc-Dec)', 'Prophet', 'LightGBM', 'Ensemble']
x, w = np.arange(4), 0.2
R2s = [compute_metrics(val_comp['volume'], val_comp[c])[0] for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
RMs = [compute_metrics(val_comp['volume'], val_comp[c])[1] for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
MAs = [compute_metrics(val_comp['volume'], val_comp[c])[2] for c in ['pred_lstm','pred_prophet','pred_gbm','pred_ensemble']]
b1 = ax4.bar(x-w, R2s, w, color='#3B82F6', alpha=0.85, label='R2')
b2 = ax4.bar(x,   [r/100 for r in RMs], w, color='#EF4444', alpha=0.85, label='RMSE/100')
b3 = ax4.bar(x+w, [m/100 for m in MAs], w, color='#10B981', alpha=0.85, label='MAE/100')
for b, v in zip(b1, R2s): ax4.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax4.axhline(0.855, color='#3B82F6', lw=1.5, ls=':', alpha=0.5, label='Makale R2=0.855')
ax4.set_xticks(x); ax4.set_xticklabels(mnames, fontsize=11)
ax4.set_title('Model Performansi — Val 2025', fontsize=12, fontweight='bold')
ax4.set_ylim(0, 1.15); ax4.legend(fontsize=9, ncol=4); ax4.grid(axis='y', alpha=0.2)
best_idx = int(np.argmax(R2s))
ax4.text(0.5, 0.92, f'En iyi: {["LSTM","Prophet","LightGBM","Ensemble"][best_idx]} (R2={R2s[best_idx]:.3f})',
         transform=ax4.transAxes, ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF3C7', alpha=0.9))

# Satir 5: Doluluk
ax5L = fig.add_subplot(gs[5, :2]); ax5L.set_facecolor('white')
ax5R = ax5L.twinx()
dol_vis = dol_full[(dol_full.index >= daily['date'].min()) & (dol_full.index <= daily['date'].max())]
ax5L.fill_between(dol_vis.index, dol_vis.values, alpha=0.15, color='#8B5CF6')
ax5L.plot(dol_vis.index, dol_vis.values, color='#8B5CF6', lw=0.8, alpha=0.7, label='Doluluk (tum)')
ax5L.plot(df_dol['doluluk'].index, df_dol['doluluk'].values, color='#4C1D95', lw=1.4, label='Doluluk (gercek)')
ax5R.plot(daily['date'], daily['volume'], color='#3B82F6', lw=0.6, alpha=0.4, label='Vaka')
ax5L.set_ylabel('Doluluk', color='#4C1D95', fontsize=9); ax5R.set_ylabel('Vaka', color='#3B82F6', fontsize=9)
ax5L.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax5L.set_title('Doluluk & Vaka Hacmi', fontsize=10, fontweight='bold')
l1, lab1 = ax5L.get_legend_handles_labels(); l2, lab2 = ax5R.get_legend_handles_labels()
ax5L.legend(l1+l2, lab1+lab2, fontsize=8)
ax5L.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax5L.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
plt.setp(ax5L.xaxis.get_majorticklabels(), rotation=30, ha='right'); ax5L.grid(alpha=0.2)

ax5S = fig.add_subplot(gs[5, 2]); ax5S.set_facecolor('white')
overlap = daily[(daily['date'] >= '2022-12-01') & (daily['volume'] > 0)].copy()
if len(overlap) > 10:
    sc5 = ax5S.scatter(overlap['doluluk'], overlap['volume'],
                       c=overlap['date'].map(pd.Timestamp.toordinal), cmap='plasma', alpha=0.35, s=18)
    m5, b5 = np.polyfit(overlap['doluluk'], overlap['volume'], 1)
    x5 = np.linspace(overlap['doluluk'].min(), overlap['doluluk'].max(), 100)
    ax5S.plot(x5, m5*x5+b5, color='#EF4444', lw=2)
    corr5 = overlap['doluluk'].corr(overlap['volume'])
    plt.colorbar(sc5, ax=ax5S)
    ax5S.text(0.05, 0.88, f'Pearson r = {corr5:.3f}', transform=ax5S.transAxes, fontsize=10,
              fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', facecolor='#EDE9FE', alpha=0.9))
ax5S.set_xlabel('Doluluk'); ax5S.set_ylabel('Vaka')
ax5S.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax5S.set_title('Doluluk x Vaka', fontsize=10, fontweight='bold'); ax5S.grid(alpha=0.2)

fig.suptitle('KUH Hastanesi - Ameliyat Vaka Hacmi Tahmini (2020-2026) v5-revised\n'
             'LSTM · Prophet · LightGBM · Weighted Ensemble · Doluluk Orani',
             fontsize=13, fontweight='bold', y=0.98)
plt.savefig('kuh_forecast_v5_revised.png', dpi=150, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print('Grafik kaydedildi: kuh_forecast_v5_revised.png')


---
## HÜCRE 14 — Model Kaydetme

**Ne yapar?** Egitilmis modelleri ve tahmin sonuclarini diske kaydeder.

| Dosya | Icerik |
|---|---|
| kuh_lstm_v5_revised.keras | Egitilmis LSTM modeli |
| kuh_lgb_scaler_v5.pkl | LightGBM + RobustScaler + ensemble agirliklari |
| kuh_tahmin_oca_mar_2026.csv | Oca-Mar 2026 tahminleri + gercek |
| kuh_validasyon_2025.csv | 2025 validasyon sonuclari |

Colab'da indirmek icin: from google.colab import files; files.download('dosya.csv')


In [ ]:
# CELL 14 — Kaydet
import pickle

lstm_model.save('kuh_lstm_v5_revised.keras')
print('LSTM kaydedildi: kuh_lstm_v5_revised.keras')

with open('kuh_lgb_scaler_v5.pkl', 'wb') as f:
    pickle.dump({
        'lgb_models': lgb_models, 'scaler': scaler, 'feature_cols': FEATURE_COLS,
        'weights': {'lstm': w_lstm, 'prophet': w_prophet, 'lgb': w_gbm},
        'params': {'LOOKBACK': LOOKBACK, 'HORIZON': HORIZON, 'COVID_END': COVID_END}
    }, f)
print('LightGBM + scaler kaydedildi: kuh_lgb_scaler_v5.pkl')

test_comp.to_csv('kuh_tahmin_oca_mar_2026.csv', index=False)
val_comp.to_csv('kuh_validasyon_2025.csv', index=False)
print('CSV ciktilar kaydedildi')

# Colab icin yorum satirini kaldir:
# from google.colab import files
# for f in ['kuh_forecast_v5_revised.png', 'kuh_tahmin_oca_mar_2026.csv',
#           'kuh_validasyon_2025.csv', 'kuh_lstm_v5_revised.keras']:
#     files.download(f)

print('\nTum dosyalar hazir!')
print(f'  Ensemble Val R2={r2e:.3f}  RMSE={rme:.1f}  MAE={mae_e:.1f}')
if has_actual:
    r2_26, rm_26, ma_26 = compute_metrics(test_comp['volume'], test_comp['pred_ensemble'])
    print(f'  Ensemble 2026 R2={r2_26:.3f}  RMSE={rm_26:.1f}  (Oca-05 Mar gercek)')
